# 🔬 MobileViT-XXS Hardware Accelerator — End-to-End Notebook

---

## 📌 Project Overview

This notebook documents the complete pipeline for the **MobileViT-XXS Hardware Accelerator** project.  
The goal is to build a **from-scratch NumPy inference engine** for MobileViT-XXS that is fully weight-compatible with Apple's official pretrained model — no deep-learning framework required at inference time.

This is the foundation for a hardware accelerator implementation, where each operation (convolution, batch norm, attention, etc.) maps directly to a hardware module.

### 🏗️ Pipeline at a Glance

```
Apple Pretrained Weights (pytorch_model.bin)
            │
            ▼
   MobileViT_XXS  ──── (MVT.py — PyTorch Reference Model)
   (PyTorch Model)
            │  load_apple_weights()  [Mapping_PretraindModel_MVT.py]
            ▼
   Weight-Mapped PyTorch Model
            │  extract_all_weights()  [Mapping_PretraindModel_NOapi.py]
            ▼
   NumPy Weight Arrays + BN Running Stats (178 tensors + 64 BN stats)
            │
            ▼
   MobileViT_XXS_()  ─── Pure NumPy Inference Engine
            │
            ▼
   Logits (1000 ImageNet classes)  ✅  matches PyTorch within 3e-4
```

---

## 🍎 About the Pretrained Model

The pretrained weights come from **Apple's official MobileViT-XXS** checkpoint (`apple/mobilevit-xx-small`), trained from scratch with the following setup:

| Setting | Value |
|---------|-------|
| Dataset | ImageNet-1k |
| Epochs | 300 |
| GPUs | 8× NVIDIA (effective batch size 1024) |
| LR Schedule | 3k-step warmup → cosine annealing |
| Loss | Label smoothing cross-entropy |
| Regularisation | L2 weight decay |
| Resolution | 160×160 → 320×320 (multi-scale sampling) |

---

## 📋 Notebook Structure

| # | Section | What it does |
|---|---------|--------------|
| 1 | **PyTorch Reference Model (MVT.py)** | Defines MobileViT-XXS in PyTorch — the ground truth |
| 2 | **Weight Mapping** | Loads Apple's checkpoint and maps to our model's keys |
| 3 | **NumPy Inference Engine** | All ops (conv, BN, attention, etc.) in pure NumPy |
| 4 | **Weight Extraction** | Converts PyTorch tensors → NumPy arrays |
| 5 | **Numerical Validation** | Runs both engines on the same input and checks agreement |


---

## 🏛️ Section 1 — MobileViT-XXS Architecture

MobileViT-XXS is a **lightweight hybrid vision transformer** that combines the local inductive bias of MobileNetV2 (MV2) blocks with the global context of Vision Transformers.  
The XXS variant is the smallest in the family, with only **~1.3M parameters**.

### Feature Map Dimensions (224×224 input)

```
Input:  (B, 3, 224, 224)
  │
  ├─ Stem:  3×3 Conv s=2  →  (B, 16, 112, 112)
  │
  ├─ Stage 1:
  │    mv2_1  [16→16, s=1]     →  (B, 16, 112, 112)
  │
  ├─ Stage 2:
  │    mv2_2a [16→24, s=2]     →  (B, 24, 56, 56)
  │    mv2_2b [24→24, s=1] ×2  →  (B, 24, 56, 56)
  │
  ├─ Stage 3:
  │    mv2_3a [24→48, s=2]     →  (B, 48, 28, 28)
  │    mvit_3b                 →  (B, 48, 28, 28)   ← Transformer (d=64, L=2, H=2)
  │
  ├─ Stage 4:
  │    mv2_4a [48→64, s=2]     →  (B, 64, 14, 14)
  │    mvit_4b                 →  (B, 64, 14, 14)   ← Transformer (d=80, L=4, H=4)
  │
  ├─ Stage 5:
  │    mv2_5a [64→80, s=2]     →  (B, 80,  7,  7)
  │    mvit_5b                 →  (B, 80,  7,  7)   ← Transformer (d=96, L=3, H=4)
  │
  ├─ Head Conv: 1×1  80→320   →  (B, 320, 7, 7)
  ├─ Global Avg Pool          →  (B, 320)
  └─ Linear Classifier        →  (B, 1000)
```

### MobileViTBlock Internal Flow

```
  Input (B, C, H, W)
      │
      ├──────────────────────────────────┐  (residual kept for fusion)
      │                                  │
  Local Representation:                  │
    3×3 Conv → BN → Swish               │
    1×1 Conv  (projects to dim d)        │
      │                                  │
  Unfold: (B,C,H,W) → (B·P, N, d)       │
      │  where P = patch_area, N = num_patches
      │                                  │
  Transformer Encoder × L layers:        │
    LN → Multi-Head Attention → skip     │
    LN → MLP (Swish) → skip             │
      │                                  │
  Final LayerNorm                        │
      │                                  │
  Fold: (B·P, N, d) → (B, d, H, W)      │
      │                                  │
  Fusion:                                │
    1×1 Conv → BN → Swish               │
    Concat with residual ◄───────────────┘
    3×3 Conv → BN → Swish
      │
  Output (B, C, H, W)
```


---

## ⚙️ Dependencies


In [ ]:
# Install required packages (run once)
# !pip install torch torchvision einops transformers


---

## 📦 Section 1 — PyTorch Reference Model (MVT.py)

This is the ground-truth implementation of **MobileViT-XXS in PyTorch**.  
It defines all the building blocks:

- **`MV2`** — MobileNetV2-style inverted residual block (expand → depthwise → project)
- **`LocalRepresentation`** — 3×3 conv + BN + Swish, then 1×1 conv (projects channels to transformer dim `d`)
- **`TransformerEncoder`** — Pre-LN transformer block with Multi-Head Attention and Swish MLP
- **`Fusion`** — Projects global features back to input channel space, concatenates with the residual, and fuses with a 3×3 conv
- **`MobileViTBlock`** — Combines LocalRepresentation + Unfold + Transformer + LayerNorm + Fold + Fusion
- **`MobileViT_XXS`** — Full model: stem → 5 stages → head conv → GAP → classifier

> **Why define it ourselves?**  
> Apple's HuggingFace model uses a different internal key naming convention.  
> By defining our own PyTorch model with explicit attribute names, we gain full control  
> over the weight mapping and can faithfully export every tensor to NumPy.


In [ ]:
## ============================================================
## PyTorch Reference Model — MobileViT-XXS (MVT.py)
## ============================================================
##
## Building blocks (bottom-up):
##   MV2               : MobileNetV2 inverted-residual block
##   LocalRepresentation: 3x3 + BN + SiLU  then  1x1 (no BN)
##   TransformerEncoder : Pre-LN MHA + Swish MLP
##   Fusion             : Project global → concat with local → 3x3 fuse
##   MobileViTBlock     : LocalRep → Unfold → Transformer → LN → Fold → Fusion
##   MobileViT_XXS      : Full end-to-end model
## ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from einops import rearrange
from torch import Tensor
from typing import Tuple, Dict
import numpy as np


# ──────────────────────────────────────────────
# MobileNetV2 Inverted Residual Block (MV2)
# ──────────────────────────────────────────────
class MV2(nn.Module):
    """
    MobileNetV2-style inverted residual block.

    Structure:
        1×1 expand (PW) → BN → SiLU
        3×3 depthwise   → BN → SiLU
        1×1 project (PW) → BN
        + residual if stride=1 and shapes match

    Args:
        in_channels    : input feature channels
        out_channels   : output feature channels
        strides        : stride for the depthwise conv (1 or 2)
        expansion_ratio: channel expansion factor for hidden dim
    """
    def __init__(self, in_channels, out_channels, strides=1, expansion_ratio=1):
        super().__init__()
        hidden_dim = int(in_channels * expansion_ratio)
        self.use_residual = strides == 1 and in_channels == out_channels

        # 1. Point-wise expand conv
        self.pw_expand = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU()
        )
        # 2. Depth-wise conv (one filter per channel, stride controls downsampling)
        self.dw_conv = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, stride=strides,
                      padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU()
        )
        # 3. Point-wise project conv (linear — no activation)
        self.pw_project = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        skip = x
        x = self.pw_expand(x)
        x = self.dw_conv(x)
        x = self.pw_project(x)
        if self.use_residual:
            x = x + skip
        return x


# ──────────────────────────────────────────────
# Local Representation
# ──────────────────────────────────────────────
class LocalRepresentation(nn.Module):
    """
    Extracts local features before the global transformer.

    Structure:
        3×3 Conv → BN → SiLU   (num_filters channels)
        1×1 Conv               (projects to transformer dim, no BN/activation)

    The 1×1 conv is intentionally kept linear — the transformer
    will apply its own normalization (LayerNorm).
    """
    def __init__(self, in_channels, num_filters, out_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, num_filters, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(num_filters)
        self.act1  = nn.SiLU()
        self.conv2 = nn.Conv2d(num_filters, out_dim, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.conv2(x)
        return x


# ──────────────────────────────────────────────
# Transformer Encoder Block
# ──────────────────────────────────────────────
class TransformerEncoder(nn.Module):
    """
    Pre-LayerNorm transformer encoder block.

    Structure (Pre-LN, i.e. normalize BEFORE each sub-block):
        LN → Multi-Head Attention → residual
        LN → FC1 → SiLU → FC2   → residual

    Args:
        dim      : embedding dimension (d_model)
        num_heads: number of attention heads
        mlp_dim  : hidden size of the feed-forward network (typically 2×dim)
    """
    def __init__(self, dim, num_heads, mlp_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads,
                                            batch_first=True, dropout=0.0)
        self.norm2 = nn.LayerNorm(dim)
        self.fc1   = nn.Linear(dim, mlp_dim)
        self.act   = nn.SiLU()
        self.fc2   = nn.Linear(mlp_dim, dim)

    def forward(self, x):
        # ── Attention sub-block ──────────────────────
        skip = x
        x, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = skip + x

        # ── MLP sub-block ────────────────────────────
        skip2 = x
        x = self.fc2(self.act(self.fc1(self.norm2(x))))
        return skip2 + x


# ──────────────────────────────────────────────
# Fusion Module
# ──────────────────────────────────────────────
class Fusion(nn.Module):
    """
    Fuses global transformer output with the local (pre-transformer) features.

    Structure:
        global_feat (dim channels)
            1×1 Conv → BN → SiLU   (projects back to in_channels)
        Concat with local residual  → (2×in_channels)
            3×3 Conv → BN → SiLU   (output: in_channels)

    Args:
        in_channels: spatial feature channels (C)
        dim        : transformer embedding dim (d_model)
    """
    def __init__(self, in_channels, dim):
        super().__init__()
        self.conv1 = nn.Conv2d(dim, in_channels, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn1   = nn.BatchNorm2d(in_channels)
        self.act1  = nn.SiLU()
        self.conv2 = nn.Conv2d(2*in_channels, in_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(in_channels)
        self.act2  = nn.SiLU()

    def forward(self, x, x_fusion):
        x_f = self.act1(self.bn1(self.conv1(x_fusion)))
        return self.act2(self.bn2(self.conv2(torch.cat([x, x_f], dim=1))))


# ──────────────────────────────────────────────
# MobileViT Block
# ──────────────────────────────────────────────
class MobileViTBlock(nn.Module):
    """
    Core MobileViT block: combines local CNN features with global Transformer attention.

    Key innovation: treats each spatial location as a token using a
    patch-based unfolding strategy rather than downsampling, preserving resolution.

    Args:
        in_channels : spatial feature channels
        num_filters : intermediate channels in LocalRepresentation
        dim         : transformer embedding dim (d_model)
        num_heads   : number of attention heads
        patch_size  : patch height = patch width (P); tokens per patch = P²
        num_layers  : number of transformer encoder layers (L)
    """
    def __init__(self, in_channels, num_filters, dim, num_heads, patch_size=2, num_layers=1):
        super().__init__()
        self.patch_h    = patch_size
        self.patch_w    = patch_size
        self.patch_area = patch_size * patch_size

        self.local_rep    = LocalRepresentation(in_channels, num_filters, dim)
        self.transformers = nn.ModuleList([
            TransformerEncoder(dim, num_heads=num_heads, mlp_dim=dim*2)
            for _ in range(num_layers)
        ])
        self.norm  = nn.LayerNorm(dim)   # applied after all transformer layers
        self.fusion = Fusion(in_channels, dim)

    def unfolding(self, x: Tensor) -> Tuple[Tensor, dict]:
        """
        Converts spatial feature map into patches for the transformer.

        (B, C, H, W) → (B·P², N, C)  where N = (H/P)·(W/P) = number of patches

        Bilinear interpolation is used if H or W is not divisible by patch_size.
        """
        b, c, h, w = x.shape
        new_h = int(np.ceil(h / self.patch_h) * self.patch_h)
        new_w = int(np.ceil(w / self.patch_w) * self.patch_w)
        interpolate = False
        if new_h != h or new_w != w:
            x = F.interpolate(x, size=(new_h, new_w), mode="bilinear", align_corners=False)
            interpolate = True

        num_patch_h = new_h // self.patch_h
        num_patch_w = new_w // self.patch_w
        num_patches = num_patch_h * num_patch_w

        x = x.reshape(b * c * num_patch_h, self.patch_h, num_patch_w, self.patch_w)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(b, c, num_patches, self.patch_area)
        x = x.permute(0, 3, 2, 1)
        patches = x.reshape(b * self.patch_area, num_patches, c)

        info = {
            "orig_size": (h, w), "batch_size": b, "interpolate": interpolate,
            "num_patches_h": num_patch_h, "num_patches_w": num_patch_w,
            "total_patches": num_patches
        }
        return patches, info

    def folding(self, x: Tensor, info: dict) -> Tensor:
        """
        Inverse of unfolding: reconstructs spatial feature map from patches.

        (B·P², N, C) → (B, C, H, W)
        """
        b            = info["batch_size"]
        num_patches_h = info["num_patches_h"]
        num_patches_w = info["num_patches_w"]
        h_new = num_patches_h * self.patch_h
        w_new = num_patches_w * self.patch_w
        channels = x.shape[-1]

        x = x.reshape(b, self.patch_area, info["total_patches"], channels)
        x = x.permute(0, 3, 2, 1)
        x = x.reshape(b * channels * num_patches_h, num_patches_w, self.patch_h, self.patch_w)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(b, channels, h_new, w_new)

        if info["interpolate"]:
            x = F.interpolate(x, size=info["orig_size"], mode="bilinear", align_corners=False)
        return x

    def forward(self, x):
        res = x                                  # save for fusion residual
        x   = self.local_rep(x)                  # local feature extraction

        patches, info = self.unfolding(x)        # → (B·P², N, d)
        for transformer in self.transformers:
            patches = transformer(patches)
        patches = self.norm(patches)             # final LayerNorm

        x_global = self.folding(patches, info)   # → (B, d, H, W)
        return self.fusion(res, x_global)        # fuse with original input


# ──────────────────────────────────────────────
# Full MobileViT-XXS Model
# ──────────────────────────────────────────────
class MobileViT_XXS(nn.Module):
    """
    MobileViT-XXS — end-to-end model.

    ~1.3M parameters, designed for mobile devices.
    Input:  (B, 3, H, W)  — recommended H=W=256
    Output: (B, num_classes)
    """
    def __init__(self, in_channels=3, num_classes=1000):
        super().__init__()

        # ── Stem ────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.SiLU()
        )

        # ── Stage 1: single MV2 (no downsampling) ───
        self.mv2_1 = MV2(16, 16, expansion_ratio=2)

        # ── Stage 2: downsample + 2 residual MV2 ────
        self.mv2_2a = MV2(16, 24, strides=2, expansion_ratio=2)
        self.mv2_2b = MV2(24, 24, expansion_ratio=2)
        self.mv2_2c = MV2(24, 24, expansion_ratio=2)

        # ── Stage 3: downsample MV2 + MobileViTBlock ─
        #    Transformer: d=64, L=2, H=2
        self.mv2_3a  = MV2(24, 48, strides=2, expansion_ratio=2)
        self.mvit_3b = MobileViTBlock(48, 48, 64, num_heads=2, patch_size=2, num_layers=2)

        # ── Stage 4: downsample MV2 + MobileViTBlock ─
        #    Transformer: d=80, L=4, H=4
        self.mv2_4a  = MV2(48, 64, strides=2, expansion_ratio=2)
        self.mvit_4b = MobileViTBlock(64, 64, 80, num_heads=4, patch_size=2, num_layers=4)

        # ── Stage 5: downsample MV2 + MobileViTBlock ─
        #    Transformer: d=96, L=3, H=4
        self.mv2_5a  = MV2(64, 80, strides=2, expansion_ratio=2)
        self.mvit_5b = MobileViTBlock(80, 80, 96, num_heads=4, patch_size=2, num_layers=3)

        # ── Head ─────────────────────────────────────
        self.head_conv = nn.Sequential(
            nn.Conv2d(80, 320, kernel_size=1, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(320),
            nn.SiLU()
        )
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.dropout    = nn.Dropout(p=0.1)
        self.classifier = nn.Linear(320, num_classes)

    def forward(self, x):
        x = self.stem(x)                                    # 224→112

        x = self.mv2_1(x)                                   # 112→112

        x = self.mv2_2a(x)                                  # 112→56
        x = self.mv2_2b(x)                                  # 56→56
        x = self.mv2_2c(x)                                  # 56→56

        x = self.mv2_3a(x)                                  # 56→28
        x = self.mvit_3b(x)                                 # 28→28 (global)

        x = self.mv2_4a(x)                                  # 28→14
        x = self.mvit_4b(x)                                 # 14→14 (global)

        x = self.mv2_5a(x)                                  # 14→7
        x = self.mvit_5b(x)                                 # 7→7  (global)

        x = self.head_conv(x)                               # 80→320
        x = self.pool(x)                                    # 7×7 → 1×1
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# ── Quick sanity check ───────────────────────────────────────────────────────
my_model = MobileViT_XXS(in_channels=3, num_classes=1000)

total_params = sum(p.numel() for p in my_model.parameters())
print(f"✅ MobileViT-XXS instantiated successfully")
print(f"   Total parameters : {total_params:,}  (~{total_params/1e6:.2f}M)")

x_test = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    y_test = my_model(x_test)
print(f"   Forward pass OK  : input {tuple(x_test.shape)} → output {tuple(y_test.shape)}")


✅ MobileViT-XXS instantiated successfully
   Total parameters : 1,272,024  (~1.27M)
   Forward pass OK  : input (2, 3, 224, 224) → output (2, 1000)


---

## 🔗 Section 2 — Weight Mapping: Apple → Our Model

Apple's pretrained MobileViT uses **HuggingFace `transformers`** key naming, which looks like:

```
mobilevit.encoder.layer.2.conv_kxk.convolution.weight
mobilevit.encoder.layer.2.transformer.layer.0.attention.attention.query.weight
...
```

Our PyTorch model uses clean, explicit attribute names:

```
mvit_3b.local_rep.conv1.weight
mvit_3b.transformers.0.attn.in_proj_weight    ← Q/K/V concatenated
...
```

### Key Challenges in the Mapping

| Challenge | Solution |
|-----------|----------|
| Attention weights split (Q, K, V separate) → PyTorch `in_proj_weight` (concatenated QKV) | `torch.cat([qw, kw, vw], dim=0)` |
| BatchNorm needs both `weight/bias` AND `running_mean/running_var` | `map_bn()` helper copies all 4 tensors |
| Apple's encoder indices (0–4) vs. our stage naming (1, 2a/b/c, 3a/b…) | Explicit per-layer mapping table |
| `strict=False` load to handle any residual mismatches | `load_state_dict(..., strict=False)` |

> **CRITICAL:** The model must be set to `eval()` mode after loading weights.  
> In training mode, BatchNorm uses batch statistics; in eval mode, it uses the  
> **running statistics** that were accumulated during Apple's training — which is what we want.


In [ ]:
## ============================================================
## Weight Mapping: Apple's pretrained checkpoint → our MVT model
## ============================================================
##
## Apple's HuggingFace naming:   mobilevit.encoder.layer.X.conv_kxk.convolution.weight
## Our naming:                   mvit_3b.local_rep.conv1.weight
##
## Key design decisions:
##   - map_bn() always copies weight, bias, running_mean, running_var
##   - QKV weights are concatenated (Apple stores them separately)
##   - strict=False allows the load to succeed even if some keys are missing
##   - model.eval() is CRITICAL — switches BN to use running stats
## ============================================================

def load_apple_weights(my_model, apple_state_dict):
    """
    Maps Apple's pretrained MobileViT-XXS checkpoint onto our custom PyTorch model.

    Apple's HuggingFace model stores weights with verbose hierarchical keys
    (e.g. mobilevit.encoder.layer.2.conv_kxk.convolution.weight).
    This function builds a new state dict with our model's keys and loads it.

    Parameters
    ----------
    my_model        : MobileViT_XXS instance (our model)
    apple_state_dict: dict loaded from pytorch_model.bin via torch.load()
    """
    new_state_dict = {}

    # ── Helper: map all 4 BatchNorm tensors at once ──────────────────────────
    # BatchNorm has 4 tensors: weight (γ), bias (β), running_mean, running_var
    # Forgetting running_mean/running_var is a common bug — this helper prevents it.
    def map_bn(my_prefix, apple_prefix):
        return {
            f'{my_prefix}.weight':       apple_state_dict[f'{apple_prefix}.weight'],
            f'{my_prefix}.bias':         apple_state_dict[f'{apple_prefix}.bias'],
            f'{my_prefix}.running_mean': apple_state_dict[f'{apple_prefix}.running_mean'],
            f'{my_prefix}.running_var':  apple_state_dict[f'{apple_prefix}.running_var'],
        }

    # ── 1. Stem ──────────────────────────────────────────────────────────────
    # Apple: mobilevit.conv_stem.{convolution, normalization}
    new_state_dict['stem.0.weight'] = apple_state_dict['mobilevit.conv_stem.convolution.weight']
    new_state_dict.update(map_bn('stem.1', 'mobilevit.conv_stem.normalization'))

    # ── 2. MV2 Block Mapping Helper ──────────────────────────────────────────
    # Each MV2 block has 3 sub-layers: expand (1×1), depthwise (3×3), project (1×1)
    def map_mv2(my_prefix, apple_prefix):
        m = {}
        # Expand 1×1 conv
        m[f'{my_prefix}.pw_expand.0.weight'] = apple_state_dict[f'{apple_prefix}.expand_1x1.convolution.weight']
        m.update(map_bn(f'{my_prefix}.pw_expand.1', f'{apple_prefix}.expand_1x1.normalization'))
        # Depthwise 3×3 conv
        m[f'{my_prefix}.dw_conv.0.weight'] = apple_state_dict[f'{apple_prefix}.conv_3x3.convolution.weight']
        m.update(map_bn(f'{my_prefix}.dw_conv.1', f'{apple_prefix}.conv_3x3.normalization'))
        # Project 1×1 conv (linear — no activation in the original)
        m[f'{my_prefix}.pw_project.0.weight'] = apple_state_dict[f'{apple_prefix}.reduce_1x1.convolution.weight']
        m.update(map_bn(f'{my_prefix}.pw_project.1', f'{apple_prefix}.reduce_1x1.normalization'))
        return m

    # Stage 1 and 2 — pure MV2 blocks
    # Apple encoder.layer.0 = Stage 1 (single block)
    # Apple encoder.layer.1 = Stage 2 (three blocks: 2a, 2b, 2c)
    new_state_dict.update(map_mv2('mv2_1',  'mobilevit.encoder.layer.0.layer.0'))
    new_state_dict.update(map_mv2('mv2_2a', 'mobilevit.encoder.layer.1.layer.0'))
    new_state_dict.update(map_mv2('mv2_2b', 'mobilevit.encoder.layer.1.layer.1'))
    new_state_dict.update(map_mv2('mv2_2c', 'mobilevit.encoder.layer.1.layer.2'))

    # Stage 3–5 — downsampling MV2 (the 'a' block before the ViT block)
    new_state_dict.update(map_mv2('mv2_3a', 'mobilevit.encoder.layer.2.downsampling_layer'))
    new_state_dict.update(map_mv2('mv2_4a', 'mobilevit.encoder.layer.3.downsampling_layer'))
    new_state_dict.update(map_mv2('mv2_5a', 'mobilevit.encoder.layer.4.downsampling_layer'))

    # ── 3. MobileViT Block Mapping ───────────────────────────────────────────
    # Each MobileViT block contains:
    #   local_rep    : conv_kxk (3×3 + BN), conv_1x1 (no BN)
    #   transformers : L transformer encoder layers
    #   norm         : final LayerNorm (applied after all transformer layers)
    #   fusion       : conv_projection (1×1), fusion (3×3)
    def map_mvit_block(my_name, apple_idx):
        m = {}
        apple_base = f'mobilevit.encoder.layer.{apple_idx}'

        # Local Representation
        m[f'{my_name}.local_rep.conv1.weight'] = apple_state_dict[f'{apple_base}.conv_kxk.convolution.weight']
        m.update(map_bn(f'{my_name}.local_rep.bn1', f'{apple_base}.conv_kxk.normalization'))
        m[f'{my_name}.local_rep.conv2.weight'] = apple_state_dict[f'{apple_base}.conv_1x1.convolution.weight']

        # Final LayerNorm after transformer stack (NOT inside each transformer layer)
        m[f'{my_name}.norm.weight'] = apple_state_dict[f'{apple_base}.layernorm.weight']
        m[f'{my_name}.norm.bias']   = apple_state_dict[f'{apple_base}.layernorm.bias']

        # Fusion: project global → local space, then concat + 3×3 fuse
        m[f'{my_name}.fusion.conv1.weight'] = apple_state_dict[f'{apple_base}.conv_projection.convolution.weight']
        m.update(map_bn(f'{my_name}.fusion.bn1', f'{apple_base}.conv_projection.normalization'))
        m[f'{my_name}.fusion.conv2.weight'] = apple_state_dict[f'{apple_base}.fusion.convolution.weight']
        m.update(map_bn(f'{my_name}.fusion.bn2', f'{apple_base}.fusion.normalization'))

        # Transformer layers
        # Apple stores Q, K, V weights separately; PyTorch MHA uses a single
        # concatenated in_proj_weight of shape (3*d, d). We must cat [Q; K; V].
        num_layers = len(getattr(my_model, my_name).transformers)
        for i in range(num_layers):
            my_t    = f'{my_name}.transformers.{i}'
            apple_t = f'{apple_base}.transformer.layer.{i}'
            attn_p  = f'{apple_t}.attention'

            # Layer Norms (Pre-LN before attention and MLP)
            m[f'{my_t}.norm1.weight'] = apple_state_dict[f'{apple_t}.layernorm_before.weight']
            m[f'{my_t}.norm1.bias']   = apple_state_dict[f'{apple_t}.layernorm_before.bias']
            m[f'{my_t}.norm2.weight'] = apple_state_dict[f'{apple_t}.layernorm_after.weight']
            m[f'{my_t}.norm2.bias']   = apple_state_dict[f'{apple_t}.layernorm_after.bias']

            # MLP (feed-forward) layers
            m[f'{my_t}.fc1.weight'] = apple_state_dict[f'{apple_t}.intermediate.dense.weight']
            m[f'{my_t}.fc1.bias']   = apple_state_dict[f'{apple_t}.intermediate.dense.bias']
            m[f'{my_t}.fc2.weight'] = apple_state_dict[f'{apple_t}.output.dense.weight']
            m[f'{my_t}.fc2.bias']   = apple_state_dict[f'{apple_t}.output.dense.bias']

            # Multi-Head Attention: concatenate Q, K, V → in_proj_weight / in_proj_bias
            qw = apple_state_dict[f'{attn_p}.attention.query.weight']
            kw = apple_state_dict[f'{attn_p}.attention.key.weight']
            vw = apple_state_dict[f'{attn_p}.attention.value.weight']
            m[f'{my_t}.attn.in_proj_weight'] = torch.cat([qw, kw, vw], dim=0)

            qb = apple_state_dict[f'{attn_p}.attention.query.bias']
            kb = apple_state_dict[f'{attn_p}.attention.key.bias']
            vb = apple_state_dict[f'{attn_p}.attention.value.bias']
            m[f'{my_t}.attn.in_proj_bias'] = torch.cat([qb, kb, vb], dim=0)

            m[f'{my_t}.attn.out_proj.weight'] = apple_state_dict[f'{attn_p}.output.dense.weight']
            m[f'{my_t}.attn.out_proj.bias']   = apple_state_dict[f'{attn_p}.output.dense.bias']
        return m

    # Map all three MobileViT blocks (stages 3b, 4b, 5b)
    new_state_dict.update(map_mvit_block('mvit_3b', 2))
    new_state_dict.update(map_mvit_block('mvit_4b', 3))
    new_state_dict.update(map_mvit_block('mvit_5b', 4))

    # ── 4. Final Head ────────────────────────────────────────────────────────
    new_state_dict['head_conv.0.weight'] = apple_state_dict['mobilevit.conv_1x1_exp.convolution.weight']
    new_state_dict.update(map_bn('head_conv.1', 'mobilevit.conv_1x1_exp.normalization'))

    # ── 5. Classifier ────────────────────────────────────────────────────────
    new_state_dict['classifier.weight'] = apple_state_dict['classifier.weight']
    new_state_dict['classifier.bias']   = apple_state_dict['classifier.bias']

    # ── Load and set eval mode ───────────────────────────────────────────────
    # strict=False allows us to proceed even if a few keys are missing.
    # eval() is CRITICAL: switches BatchNorm from batch statistics → running statistics.
    my_model.load_state_dict(new_state_dict, strict=False)
    my_model.eval()
    print("✅ Successfully mapped Apple weights and set model to eval mode.")


### Load the Pretrained Checkpoint

We load the official Apple `pytorch_model.bin` checkpoint using `torch.load()`.  
The HuggingFace `MobileViTForImageClassification` is used purely to verify the checkpoint  
is correct — the weights are immediately re-mapped into our custom model.

> **Note:** Place `pytorch_model.bin` in the same directory as this notebook,  
> or adjust the path below.


In [ ]:
from transformers import MobileViTForImageClassification

# Load the official Apple checkpoint
# HuggingFace downloads this automatically from the model hub on first run,
# or you can provide a local path to pytorch_model.bin.
apple_model      = MobileViTForImageClassification.from_pretrained("apple/mobilevit-xx-small")
apple_state_dict = torch.load("pytorch_model.bin", map_location="cpu")
apple_model.load_state_dict(apple_state_dict)

# Map Apple's weights into our custom MobileViT_XXS model
load_apple_weights(my_model, apple_state_dict)

print(f"\nApple checkpoint keys (first 5):")
for k in list(apple_state_dict.keys())[:5]:
    print(f"  {k}")
print("  ...")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/347 [00:00<?, ?it/s]

✅ Successfully mapped Apple weights and set model to eval mode.

Apple checkpoint keys (first 5):
  mobilevit.conv_stem.convolution.weight
  mobilevit.conv_stem.normalization.weight
  mobilevit.conv_stem.normalization.bias
  mobilevit.conv_stem.normalization.running_mean
  mobilevit.conv_stem.normalization.running_var
  ...


---

## 🔢 Section 3 — Pure NumPy Inference Engine

This section implements **every operation** in MobileViT-XXS using only NumPy.  
No PyTorch, no BLAS abstractions — just raw array math.

### Why NumPy?

This NumPy implementation is the **direct precursor to the hardware accelerator**.  
Each function below maps one-to-one to a hardware module:

| NumPy Function | Hardware Module |
|----------------|----------------|
| `conv_forward` | Convolution Engine |
| `depthwise_conv` | Depthwise Conv Engine |
| `batch_norm_forward` | BN Normalizer |
| `layer_norm_forward` | LN Normalizer |
| `MultiHeadAttention` | Attention Datapath |
| `MLP` | Linear Compute Unit |
| `unfold` / `fold` | Patch Shuffler |
| `fusion` | Feature Fusion Unit |

### Numerical Precision

All operations use `float32`. The final comparison against PyTorch shows:

```
Max  absolute difference : 3.3e-4
Mean absolute difference : 1.0e-4
PASS — within 5e-4 threshold ✅
```

The small differences arise from floating-point reordering in matrix multiplications.

---

### 🔍 Primitive Helper Functions


In [ ]:
## ============================================================
## Pure NumPy Inference Engine — MobileViT-XXS
## ============================================================
##
## Every operation is implemented from scratch in NumPy.
## These functions correspond directly to hardware compute units
## in the accelerator design.
##
## Naming convention:
##   W_*    : weight tensor
##   b_*    : bias tensor
##   bn_prams[i] : BN running mean (even i) or variance (odd i)
## ============================================================

import numpy as np
import math


# ── Zero-padding ─────────────────────────────────────────────────────────────
# Used before convolutions to control output spatial dimensions.
# Output shape: (m, C, H+2*pad, W+2*pad)
def zero_pad(X, pad):
    """
    Pad all images in batch X with zeros on height and width dimensions.

    Args:
        X   : (m, C, H, W)
        pad : integer padding on all four sides
    Returns:
        X_pad : (m, C, H+2·pad, W+2·pad)
    """
    return np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), mode='constant', constant_values=0)


# ── Single convolution step ───────────────────────────────────────────────────
# Applies one filter to one spatial slice. Used inside conv_forward loops.
def conv_single_step(a_slice_prev, W, b):
    """
    Apply one filter W (and bias b) to a single input slice.

    Args:
        a_slice_prev : (C_prev, f, f) — input patch
        W            : (C_prev, f, f) — filter weights
        b            : scalar bias
    Returns:
        Z : scalar output value
    """
    return float(np.sum(a_slice_prev * W) + b.item())


# ── Standard Convolution ──────────────────────────────────────────────────────
# Used for: stem, LocalRep 3×3, LocalRep 1×1, MV2 expand/project, head conv, fusion convs
def conv_forward(A_prev, W, b, stride, pad):
    """
    Forward pass: standard 2D convolution.

    Args:
        A_prev : (m, C_in, H_in, W_in)  — input feature map
        W      : (C_out, C_in, f, f)    — filter weights
        b      : (C_out, 1, 1, 1)       — biases
        stride : int
        pad    : int (zero-padding on each side)
    Returns:
        Z : (m, C_out, H_out, W_out)
    """
    m, n_C_prev, n_H_prev, n_W_prev = A_prev.shape
    f   = W.shape[2]
    n_C = W.shape[0]
    n_H = int((n_H_prev - f + 2*pad) / stride + 1)
    n_W = int((n_W_prev - f + 2*pad) / stride + 1)
    Z   = np.zeros((m, n_C, n_H, n_W))
    A_prev_pad = zero_pad(A_prev, pad)
    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        for h in range(n_H):
            vs, ve = h*stride, h*stride+f
            for w in range(n_W):
                hs, he = w*stride, w*stride+f
                for c in range(n_C):
                    Z[i,c,h,w] = conv_single_step(a_prev_pad[:, vs:ve, hs:he], W[c], b[c])
    return Z


# ── Depthwise Convolution ─────────────────────────────────────────────────────
# Each input channel is convolved with its OWN filter (groups=C).
# Used in MV2's middle 3×3 conv.
def depthwise_conv(A_prev, W, b, stride, pad):
    """
    Depthwise 2D convolution: channel i is convolved with filter W[i] only.

    Args:
        A_prev : (m, C, H_in, W_in)
        W      : (C, f, f)           — one filter per input channel
        b      : (C,)
        stride : int
        pad    : int
    Returns:
        A : (m, C, H_out, W_out)
    """
    m, n_C_prev, n_H_prev, n_W_prev = A_prev.shape
    f   = W.shape[2]
    n_H = int((n_H_prev - f + 2*pad) / stride + 1)
    n_W = int((n_W_prev - f + 2*pad) / stride + 1)
    A   = np.zeros((m, n_C_prev, n_H, n_W))
    A_prev_p = zero_pad(A_prev, pad)
    for i in range(m):
        a_prev = A_prev_p[i]
        for h in range(n_H):
            vs, ve = h*stride, h*stride+f
            for w in range(n_W):
                hs, he = w*stride, w*stride+f
                for c in range(n_C_prev):
                    A[i,c,h,w] = np.sum(a_prev[c, vs:ve, hs:he] * W[c]) + b[c]
    return A


# ── Batch Normalization (Inference Mode) ──────────────────────────────────────
# Uses RUNNING statistics (not batch statistics) — only valid after training.
# Formula: γ · (z - μ_run) / √(σ²_run + ε) + β
def batch_norm_forward(Z, gamma, beta, running_mean, running_var, eps=1e-5):
    """
    Batch normalization — inference mode (uses stored running statistics).

    Args:
        Z            : (m, C, H, W)
        gamma        : (C,) — learnable scale γ
        beta         : (C,) — learnable shift β
        running_mean : (C,) — accumulated during training
        running_var  : (C,) — accumulated during training
        eps          : float — numerical stability constant
    Returns:
        Z_norm : (m, C, H, W) — normalized, scaled, shifted
    """
    gamma = gamma.reshape(1,-1,1,1)
    beta  = beta.reshape(1,-1,1,1)
    mean  = running_mean.reshape(1,-1,1,1)
    var   = running_var.reshape(1,-1,1,1)
    return gamma * (Z - mean) / np.sqrt(var + eps) + beta


# ── Layer Normalization ───────────────────────────────────────────────────────
# Applied inside the Transformer. Normalizes over the embedding dimension (last axis).
# Unlike BatchNorm, LayerNorm does NOT use running statistics.
def layer_norm_forward(X, gamma, beta, eps=1e-5):
    """
    Layer normalization over the last dimension (embedding/channel dim).

    Args:
        X     : (..., d_model)
        gamma : (d_model,)
        beta  : (d_model,)
        eps   : float
    Returns:
        out : same shape as X
    """
    mean = np.mean(X, axis=-1, keepdims=True)
    var  = np.var(X,  axis=-1, keepdims=True)
    return gamma * (X - mean) / np.sqrt(var + eps) + beta


# ── Activation Functions ──────────────────────────────────────────────────────
# SiLU (Swish): x · σ(x). Smoother than ReLU, used throughout MobileViT.
def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

def swish(x):
    """Swish / SiLU activation: x · sigmoid(x). Same as torch.nn.SiLU."""
    return x * sigmoid(x)

def softmax(x, axis=-1):
    """Numerically stable softmax. Used for attention score normalization."""
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)


print("✅ Primitive helpers defined: zero_pad, conv_forward, depthwise_conv,")
print("   batch_norm_forward, layer_norm_forward, swish, softmax")


✅ Primitive helpers defined: zero_pad, conv_forward, depthwise_conv,
   batch_norm_forward, layer_norm_forward, swish, softmax


### 🔲 MobileNetV2 Block (MV2)


In [ ]:
# ── MobileNetV2 Inverted Residual Block (MV2) ─────────────────────────────────
# Structure:
#   1×1 expand → BN → Swish
#   3×3 depthwise → BN → Swish
#   1×1 project → BN  (+ residual if stride=1 and shapes match)
#
# This is the main building block for the CNN backbone stages.
# Hardware: two 1×1 conv engines + one depthwise engine + three BN normalizers.

def MobileNet_v2(Input,
                 W_expan, W_depth, W_point,
                 b_expan, b_depth, b_point,
                 stride_depth,
                 W_gamma_expan, W_gamma_depth, W_gamma_point,
                 b_gamma_expan, b_gamma_depth, b_gamma_point,
                 running_mean1, running_var1,   # BN after expand 1×1
                 running_mean2, running_var2,   # BN after depthwise 3×3
                 running_mean3, running_var3):  # BN after project 1×1
    """
    MobileNetV2 inverted residual block (NumPy version).

    Args:
        Input          : (m, C_in, H, W)
        W_expan/depth/point : conv weights for each sub-layer
        b_*            : corresponding biases
        stride_depth   : stride for the 3×3 depthwise (1 = no downsample, 2 = halve H/W)
        W_gamma_*, b_gamma_* : BN scale (γ) and shift (β)
        running_mean*, running_var* : BN running statistics
    Returns:
        output : (m, C_out, H_out, W_out)
    """
    # 1×1 Expand
    Z1 = conv_forward(Input, W_expan, b_expan, 1, 0)
    Z1 = batch_norm_forward(Z1, W_gamma_expan, b_gamma_expan, running_mean1, running_var1)
    A1 = swish(Z1)

    # 3×3 Depthwise
    Z2 = depthwise_conv(A1, W_depth, b_depth, stride_depth, 1)
    Z2 = batch_norm_forward(Z2, W_gamma_depth, b_gamma_depth, running_mean2, running_var2)
    A2 = swish(Z2)

    # 1×1 Project (linear — no activation in original MobileNetV2)
    Z3 = conv_forward(A2, W_point, b_point, 1, 0)
    Z3 = batch_norm_forward(Z3, W_gamma_point, b_gamma_point, running_mean3, running_var3)

    # Residual connection (only when stride=1 and dimensions match)
    if stride_depth == 1 and Input.shape == Z3.shape:
        return Z3 + Input
    return Z3


print("✅ MobileNet_v2 block defined")


✅ MobileNet_v2 block defined


### 🤖 Transformer Components (Multi-Head Attention + MLP)


In [ ]:
# ── Multi-Head Self-Attention ─────────────────────────────────────────────────
# Implements scaled dot-product attention with multiple heads.
#
# Implementation note on QKV weights:
#   PyTorch MHA stores a combined in_proj_weight of shape (3·d, d).
#   We compute QKV in one matmul, then split: Q=[:d], K=[d:2d], V=[2d:].
#   This is equivalent to Apple's separate Q/K/V projections.

def MultiHeadAttention(X, W, W_o, Bias, Bias_o, num_heads):
    """
    Multi-head self-attention.

    Args:
        X        : (B, N, d) — input tokens; B = batch*patch_area, N = num_patches
        W        : (3d, d)   — combined QKV projection weight
        W_o      : (d, d)    — output projection weight
        Bias     : (3d,)     — QKV bias
        Bias_o   : (d,)      — output projection bias
        num_heads: int
    Returns:
        out : (B, N, d)

    Steps:
        1. Linear project → QKV
        2. Split into Q, K, V and reshape for multi-head
        3. Scaled dot-product: softmax(Q·Kᵀ / √d_head) · V
        4. Concatenate heads + output projection
    """
    batch_size, seq_len, d_model = X.shape
    head_dim = d_model // num_heads

    # Project to Q, K, V (single matmul — efficient)
    QKV = np.matmul(X, W.T) + Bias.reshape(1, 1, -1)
    Q = QKV[:, :, :d_model]
    K = QKV[:, :, d_model:2*d_model]
    V = QKV[:, :, 2*d_model:]

    # Reshape to (B, heads, N, head_dim)
    Q = Q.reshape(batch_size, seq_len, num_heads, head_dim).transpose(0,2,1,3)
    K = K.reshape(batch_size, seq_len, num_heads, head_dim).transpose(0,2,1,3)
    V = V.reshape(batch_size, seq_len, num_heads, head_dim).transpose(0,2,1,3)

    # Scaled dot-product attention
    Q = Q / np.sqrt(head_dim)                           # scale
    scores  = np.matmul(Q, K.transpose(0,1,3,2))        # (B, heads, N, N)
    attn    = softmax(scores, axis=-1)                   # attention weights
    context = np.matmul(attn, V)                         # (B, heads, N, head_dim)

    # Concatenate heads and project output
    context = context.transpose(0,2,1,3).reshape(batch_size, seq_len, d_model)
    return np.matmul(context, W_o.T) + Bias_o.reshape(1, 1, -1)


# ── MLP (Feed-Forward Network) ────────────────────────────────────────────────
# Two linear layers with Swish activation in between.
# Hidden dimension = 2 × embedding dimension (MobileViT default).
def MLP(X, W1_fc1, b1_fc1, W2_fc2, b2_fc2):
    """
    Two-layer MLP with Swish activation.

    Args:
        X      : (B, N, d_model)
        W1_fc1 : (d_ff, d_model) — first layer weights
        b1_fc1 : (d_ff,)
        W2_fc2 : (d_model, d_ff) — second layer weights
        b2_fc2 : (d_model,)
    Returns:
        out : (B, N, d_model)
    """
    Z1 = np.matmul(X, W1_fc1.T) + b1_fc1   # expand to d_ff
    A1 = swish(Z1)
    return np.matmul(A1, W2_fc2.T) + b2_fc2 # project back to d_model


# ── Transformer Encoder Block ─────────────────────────────────────────────────
# Pre-LayerNorm variant: normalize BEFORE each sub-block (not after).
# This is more stable than post-LN during training.
def transformer_encoder(X,
                         W_QKV_atten, W_o_atten, Bias_QKV_atten, Bias_o_atten,
                         W1_fc1, b1_fc1, W2_fc2, b2_fc2,
                         W_gamma1, b_beta1, W_gamma2, b_beta2,
                         num_heads):
    """
    Single Pre-LN Transformer encoder block.

    Structure:
        x = x + Attention(LayerNorm(x))   ← attention sub-block
        x = x + MLP(LayerNorm(x))         ← MLP sub-block

    Args:
        X         : (B, N, d_model)
        W_QKV_atten, W_o_atten : attention weights
        Bias_*    : attention biases
        W1_fc1/W2_fc2, b1/b2 : MLP weights and biases
        W_gamma1/2, b_beta1/2 : LayerNorm parameters (γ, β)
        num_heads : number of attention heads
    Returns:
        X : (B, N, d_model)
    """
    # ── Attention sub-block (Pre-LN) ──────────────────────────────────────────
    skip1 = X
    X = MultiHeadAttention(
        layer_norm_forward(X, W_gamma1, b_beta1),
        W_QKV_atten, W_o_atten, Bias_QKV_atten, Bias_o_atten, num_heads)
    X = X + skip1

    # ── MLP sub-block (Pre-LN) ────────────────────────────────────────────────
    skip2 = X
    X = MLP(layer_norm_forward(X, W_gamma2, b_beta2), W1_fc1, b1_fc1, W2_fc2, b2_fc2)
    return X + skip2


print("✅ Transformer components defined: MultiHeadAttention, MLP, transformer_encoder")


✅ Transformer components defined: MultiHeadAttention, MLP, transformer_encoder


### 🧩 Patch Unfolding & Folding (The Core MobileViT Innovation)


In [ ]:
# ── Bilinear Resize ───────────────────────────────────────────────────────────
# Matches PyTorch's F.interpolate(..., mode='bilinear', align_corners=False).
# Used when the feature map H or W is not divisible by patch_size.
def bilinear_resize(x, new_h, new_w):
    """
    Bilinear interpolation — matches PyTorch F.interpolate(align_corners=False).

    The half-pixel convention (align_corners=False) is critical for
    numerical agreement with PyTorch. Without this, the fold/unfold
    cycle would introduce errors up to ~1e-2.
    """
    b, c, h, w = x.shape
    # Half-pixel grid: map output pixel i to input coordinate (i+0.5)*(h/new_h)-0.5
    row_idx = (np.arange(new_h) + 0.5) * (h / new_h) - 0.5
    col_idx = (np.arange(new_w) + 0.5) * (w / new_w) - 0.5
    row_idx = np.clip(row_idx, 0, h - 1)
    col_idx = np.clip(col_idx, 0, w - 1)
    r0 = np.floor(row_idx).astype(int);  r1 = np.minimum(r0 + 1, h - 1)
    c0 = np.floor(col_idx).astype(int);  c1 = np.minimum(c0 + 1, w - 1)
    dr = (row_idx - r0)[:, None]
    dc = (col_idx - c0)[None, :]
    # Bilinear interpolation formula: weighted average of 4 nearest neighbors
    top_left     = x[:, :, r0, :][:, :, :, c0]
    top_right    = x[:, :, r0, :][:, :, :, c1]
    bottom_left  = x[:, :, r1, :][:, :, :, c0]
    bottom_right = x[:, :, r1, :][:, :, :, c1]
    out = (top_left     * (1-dr) * (1-dc) +
           top_right    * (1-dr) *    dc  +
           bottom_left  *    dr  * (1-dc) +
           bottom_right *    dr  *    dc)
    return out.astype(x.dtype)


# ── Unfold: Spatial → Patches ─────────────────────────────────────────────────
# Rearranges a (B, C, H, W) feature map into tokens for the transformer.
#
# Key idea: each "patch" is a P×P region of the feature map.
#   N = (H/P)·(W/P) patches   (each patch becomes a sequence position)
#   Sequence length = N        (number of spatial locations globally)
#   Token dimension = C        (channel = embedding)
#   Batch for transformer = B·P²  (each pixel within a patch becomes a separate "batch")
#
# This clever arrangement means the transformer models interactions between
# corresponding pixels ACROSS ALL PATCHES simultaneously.
def unfold(x, patch_size=2):
    """
    Unfold spatial feature map into patches for transformer processing.

    (B, C, H, W) → (B·P², N, C)
    where P = patch_size, N = (H/P)·(W/P)

    The sequence dimension N = number of patches.
    The batch dimension B·P² means each pixel position within a patch
    is processed as a separate sequence in parallel.

    Args:
        x          : (B, C, H, W)
        patch_size : int — P (height and width of each patch)
    Returns:
        patches : (B·P², N, C)
        info    : dict with shape info needed for fold()
    """
    b, c, h, w = x.shape
    ph = pw = patch_size

    # Pad to next multiple of patch_size (if needed)
    new_h = math.ceil(h / ph) * ph
    new_w = math.ceil(w / pw) * pw
    interpolate = False
    if new_h != h or new_w != w:
        x = bilinear_resize(x, new_h, new_w)
        interpolate = True

    patch_area  = ph * pw
    num_patch_h = new_h // ph
    num_patch_w = new_w // pw
    num_patches = num_patch_h * num_patch_w

    # Reshape + transpose to interleave patches and spatial positions
    x = x.reshape(b * c * num_patch_h, ph, num_patch_w, pw)
    x = x.transpose(0, 2, 1, 3)                         # → [B*C*n_h, n_w, p_h, p_w]
    x = x.reshape(b, c, num_patches, patch_area)
    x = x.transpose(0, 3, 2, 1)                         # → [B, P², N, C]
    patches = x.reshape(b * patch_area, num_patches, c) # → [B·P², N, C]

    info = dict(b=b, c=c, orig_h=h, orig_w=w,
                num_patch_h=num_patch_h, num_patch_w=num_patch_w,
                num_patches=num_patches, patch_area=patch_area,
                ph=ph, pw=pw, interpolate=interpolate)
    return patches, info


# ── Fold: Patches → Spatial ───────────────────────────────────────────────────
# Exact inverse of unfold. Reconstructs the spatial feature map
# after the transformer has processed all patches.
def fold(patches, info):
    """
    Reconstruct spatial feature map from transformer-processed patches.

    (B·P², N, C) → (B, C, H, W)

    Args:
        patches : (B·P², N, C)  — transformer output
        info    : dict from unfold()
    Returns:
        x : (B, C, H, W)
    """
    b           = info['b']
    num_patch_h = info['num_patch_h']
    num_patch_w = info['num_patch_w']
    patch_area  = info['patch_area']
    ph, pw      = info['ph'], info['pw']
    channels    = patches.shape[-1]

    x = patches.reshape(b, patch_area, info['num_patches'], channels)
    x = x.transpose(0, 3, 2, 1)
    x = x.reshape(b * channels * num_patch_h, num_patch_w, ph, pw)
    x = x.transpose(0, 2, 1, 3)
    x = x.reshape(b, channels, num_patch_h * ph, num_patch_w * pw)

    # Revert interpolation if the feature map was padded during unfold
    if info['interpolate']:
        x = bilinear_resize(x, info['orig_h'], info['orig_w'])
    return x


print("✅ Patch operations defined: bilinear_resize, unfold, fold")
print()
print("Unfold/fold test (verifying round-trip consistency):")
test_x = np.random.randn(1, 48, 28, 28).astype(np.float32)
patches, info = unfold(test_x, patch_size=2)
reconstructed = fold(patches, info)
print(f"  Input shape    : {test_x.shape}")
print(f"  Patches shape  : {patches.shape}  (B·P²={info['patch_area']}, N={info['num_patches']}, C=48)")
print(f"  Folded shape   : {reconstructed.shape}")
print(f"  Max round-trip error: {np.abs(test_x - reconstructed).max():.2e}  (should be ~0)")


✅ Patch operations defined: bilinear_resize, unfold, fold

Unfold/fold test (verifying round-trip consistency):
  Input shape    : (1, 48, 28, 28)
  Patches shape  : (4, 196, 48)  (B·P²=4, N=196, C=48)
  Folded shape   : (1, 48, 28, 28)
  Max round-trip error: 0.00e+00  (should be ~0)


### 🧠 Local Representation, Fusion & MobileViT Block


In [ ]:
# ── Local Representation ──────────────────────────────────────────────────────
# Extracts local features from the spatial feature map before patching.
# 3×3 conv (with BN + Swish) captures local texture/edge information.
# 1×1 conv projects channels to the transformer embedding dimension d.
def Local_representations(input,
                           W_loacal_3x3, b_local_3x3,
                           W_loacal_1x1, b_local_1x1,
                           W_gamma_local_3x3, W_beta_local_3x3,
                           running_mean1, running_var1):
    """
    Local feature extraction for MobileViTBlock.

    Structure:
        3×3 Conv → BN → Swish  (captures local spatial context)
        1×1 Conv               (projects to transformer dim d, no BN/activation)

    The 1×1 conv has no normalization — the transformer uses its own LayerNorm.
    """
    # 3×3 conv → BN → Swish (local feature extraction)
    x = conv_forward(input, W_loacal_3x3, b_local_3x3, 1, 1)
    x = batch_norm_forward(x, W_gamma_local_3x3, W_beta_local_3x3, running_mean1, running_var1)
    x = swish(x)
    # 1×1 projection to transformer dim (linear, no BN)
    x = conv_forward(x, W_loacal_1x1, b_local_1x1, 1, 0)
    return x


# ── Fusion ────────────────────────────────────────────────────────────────────
# Combines the global transformer output with the original local features.
# This is what allows MobileViT to have both local AND global context.
def fusion(x_global, input_feat,
           W_fusion_1x1, b_fusion_1x1,
           W_fusion_3x3, b_fusion_3x3,
           W_gamma_f_1x1, b_beta_f_1x1,
           W_gamma_f_3x3, b_beta_f_3x3,
           running_mean1, running_var1,
           running_mean2, running_var2):
    """
    Fusion of global transformer features with local CNN features.

    Structure:
        global_feat (d channels)
            1×1 Conv → BN → Swish   ← project back to C channels
        Concat [local, projected_global]   → (2C channels)
            3×3 Conv → BN → Swish   ← fuse both streams

    Args:
        x_global   : (B, d, H, W)  — transformer output (folded)
        input_feat : (B, C, H, W)  — original feature map (before local_rep)
    Returns:
        x_fusion : (B, C, H, W)
    """
    # Project global features back to C channels
    x = conv_forward(x_global, W_fusion_1x1, b_fusion_1x1, 1, 0)
    x = batch_norm_forward(x, W_gamma_f_1x1, b_beta_f_1x1, running_mean1, running_var1)
    x = swish(x)

    if x.shape[2:] != input_feat.shape[2:]:
        raise ValueError(f"Spatial mismatch in fusion: global {x.shape[2:]} vs local {input_feat.shape[2:]}")

    # Concatenate local and projected global features along channel dim
    concat = np.concatenate([input_feat, x], axis=1)

    # 3×3 conv to fuse both streams
    x = conv_forward(concat, W_fusion_3x3, b_fusion_3x3, 1, 1)
    x = batch_norm_forward(x, W_gamma_f_3x3, b_beta_f_3x3, running_mean2, running_var2)
    return swish(x)


# ── MobileViTBlock (NumPy) ────────────────────────────────────────────────────
# Full MobileViT block: local rep → unfold → transformers → LN → fold → fusion.
def MobileViTBlock_(input,
                    W_loacal_3x3, b_local_3x3,
                    W_loacal_1x1, b_local_1x1,
                    W_gamma_local_3x3, W_beta_local_3x3,
                    W_QKV_atten, W_o_atten, Bias_QKV_atten, Bias_o_atten,
                    W1_fc1, b1_fc1, W2_fc2, b2_fc2,
                    W_gamma1, b_beta1, W_gamma2, b_beta2,
                    W_norm_gamma, b_norm_beta,        # final LayerNorm
                    W_fusion_1x1, b_fusion_1x1,
                    W_fusion_3x3, b_fusion_3x3,
                    W_gamma_f_1x1, b_beta_f_1x1,
                    W_gamma_f_3x3, b_beta_f_3x3,
                    L, patch_size, num_heads,         # block config
                    running_mean1, running_var1,       # local 3×3 BN
                    running_mean2, running_var2,       # fusion 1×1 BN
                    running_mean3, running_var3):      # fusion 3×3 BN
    """
    Complete MobileViT block (NumPy).

    Steps:
        1. Local feature extraction (3×3 + 1×1 conv)
        2. Unfold spatial → patches
        3. L × transformer encoder layers
        4. Final LayerNorm
        5. Fold patches → spatial
        6. Fusion with original input

    Args:
        input      : (B, C, H, W) — input feature map
        L          : number of transformer layers
        patch_size : patch height = patch width
        num_heads  : number of attention heads
    Returns:
        x_out : (B, C, H, W)
    """
    # Step 1: Local features (preserves input for fusion)
    x_local = Local_representations(
        input,
        W_loacal_3x3, b_local_3x3,
        W_loacal_1x1, b_local_1x1,
        W_gamma_local_3x3, W_beta_local_3x3,
        running_mean1, running_var1)

    # Step 2: Unfold → (B·P², N, d)
    patches, info = unfold(x_local, patch_size)

    # Step 3: L transformer encoder layers
    for i in range(L):
        patches = transformer_encoder(
            patches,
            W_QKV_atten[i], W_o_atten[i], Bias_QKV_atten[i], Bias_o_atten[i],
            W1_fc1[i], b1_fc1[i], W2_fc2[i], b2_fc2[i],
            W_gamma1[i], b_beta1[i], W_gamma2[i], b_beta2[i],
            num_heads)

    # Step 4: Final LayerNorm (applied after the full transformer stack)
    patches = layer_norm_forward(patches, W_norm_gamma, b_norm_beta)

    # Step 5: Fold → (B, d, H, W)
    x_global = fold(patches, info)

    # Step 6: Fusion (combines transformer output with original input)
    return fusion(
        x_global, input,
        W_fusion_1x1, b_fusion_1x1,
        W_fusion_3x3, b_fusion_3x3,
        W_gamma_f_1x1, b_beta_f_1x1,
        W_gamma_f_3x3, b_beta_f_3x3,
        running_mean2, running_var2,
        running_mean3, running_var3)


print("✅ MobileViTBlock_ defined (Local Rep + Unfold + Transformer×L + LN + Fold + Fusion)")


✅ MobileViTBlock_ defined (Local Rep + Unfold + Transformer×L + LN + Fold + Fusion)


### 🎯 Full Model: MobileViT_XXS_ (NumPy)


In [ ]:
# ── Global Average Pooling ────────────────────────────────────────────────────
def global_avg_pool(x):
    """Global average pool: (m, C, H, W) → (m, C)."""
    return np.mean(x, axis=(2, 3))

# ── Linear Classifier ─────────────────────────────────────────────────────────
def linear_classifier(x, W_cls, b_cls):
    """Linear layer: (m, feat) × (feat, classes) + bias → (m, classes)."""
    return np.matmul(x, W_cls) + b_cls


# ── Full MobileViT-XXS (NumPy) ────────────────────────────────────────────────
# This function mirrors the PyTorch model's forward() method exactly.
# Each stage is annotated with expected feature map shapes.
def MobileViT_XXS_(Inputs, bn_prams,
                   # Stem
                   W_stem, b_stem, W_stem_gamma, b_beta_stem,
                   # Stage 1
                   W_expan_1,  W_depth_1,  W_point_1,
                   b_expan_1,  b_depth_1,  b_point_1,
                   W_gamma_expan_1, W_gamma_depth_1, W_gamma_point_1,
                   b_gamma_expan_1, b_gamma_depth_1, b_gamma_point_1,
                   # Stage 2a
                   W_expan_2a, W_depth_2a, W_point_2a,
                   b_expan_2a, b_depth_2a, b_point_2a,
                   W_gamma_expan_2a, W_gamma_depth_2a, W_gamma_point_2a,
                   b_gamma_expan_2a, b_gamma_depth_2a, b_gamma_point_2a,
                   # Stage 2b
                   W_expan_2b, W_depth_2b, W_point_2b,
                   b_expan_2b, b_depth_2b, b_point_2b,
                   W_gamma_expan_2b, W_gamma_depth_2b, W_gamma_point_2b,
                   b_gamma_expan_2b, b_gamma_depth_2b, b_gamma_point_2b,
                   # Stage 2c
                   W_expan_2c, W_depth_2c, W_point_2c,
                   b_expan_2c, b_depth_2c, b_point_2c,
                   W_gamma_expan_2c, W_gamma_depth_2c, W_gamma_point_2c,
                   b_gamma_expan_2c, b_gamma_depth_2c, b_gamma_point_2c,
                   # Stage 3a
                   W_expan_3a, W_depth_3a, W_point_3a,
                   b_expan_3a, b_depth_3a, b_point_3a,
                   W_gamma_expan_3a, W_gamma_depth_3a, W_gamma_point_3a,
                   b_gamma_expan_3a, b_gamma_depth_3a, b_gamma_point_3a,
                   # Stage 3b (MobileViTBlock: d=64, L=2, H=2)
                   W_loacal_3x3_3b, b_local_3x3_3b,
                   W_loacal_1x1_3b, b_local_1x1_3b,
                   W_gamma_local_3x3_3b, W_beta_local_3x3_3b,
                   W_QKV_atten_3b, W_o_atten_3b, Bias_QKV_atten_3b, Bias_o_atten_3b,
                   W1_fc1_3b, b1_fc1_3b, W2_fc2_3b, b2_fc2_3b,
                   W_gamma1_3b, b_beta1_3b, W_gamma2_3b, b_beta2_3b,
                   W_norm_gamma_3b, b_norm_beta_3b,
                   W_fusion_1x1_3b, b_fusion_1x1_3b,
                   W_fusion_3x3_3b, b_fusion_3x3_3b,
                   W_gamma_f_1x1_3b, b_beta_f_1x1_3b,
                   W_gamma_f_3x3_3b, b_beta_f_3x3_3b,
                   # Stage 4a
                   W_expan_4a, W_depth_4a, W_point_4a,
                   b_expan_4a, b_depth_4a, b_point_4a,
                   W_gamma_expan_4a, W_gamma_depth_4a, W_gamma_point_4a,
                   b_gamma_expan_4a, b_gamma_depth_4a, b_gamma_point_4a,
                   # Stage 4b (MobileViTBlock: d=80, L=4, H=4)
                   W_loacal_3x3_4b, b_local_3x3_4b,
                   W_loacal_1x1_4b, b_local_1x1_4b,
                   W_gamma_local_3x3_4b, W_beta_local_3x3_4b,
                   W_QKV_atten_4b, W_o_atten_4b, Bias_QKV_atten_4b, Bias_o_atten_4b,
                   W1_fc1_4b, b1_fc1_4b, W2_fc2_4b, b2_fc2_4b,
                   W_gamma1_4b, b_beta1_4b, W_gamma2_4b, b_beta2_4b,
                   W_norm_gamma_4b, b_norm_beta_4b,
                   W_fusion_1x1_4b, b_fusion_1x1_4b,
                   W_fusion_3x3_4b, b_fusion_3x3_4b,
                   W_gamma_f_1x1_4b, b_beta_f_1x1_4b,
                   W_gamma_f_3x3_4b, b_beta_f_3x3_4b,
                   # Stage 5a
                   W_expan_5a, W_depth_5a, W_point_5a,
                   b_expan_5a, b_depth_5a, b_point_5a,
                   W_gamma_expan_5a, W_gamma_depth_5a, W_gamma_point_5a,
                   b_gamma_expan_5a, b_gamma_depth_5a, b_gamma_point_5a,
                   # Stage 5b (MobileViTBlock: d=96, L=3, H=4)
                   W_loacal_3x3_5b, b_local_3x3_5b,
                   W_loacal_1x1_5b, b_local_1x1_5b,
                   W_gamma_local_3x3_5b, W_beta_local_3x3_5b,
                   W_QKV_atten_5b, W_o_atten_5b, Bias_QKV_atten_5b, Bias_o_atten_5b,
                   W1_fc1_5b, b1_fc1_5b, W2_fc2_5b, b2_fc2_5b,
                   W_gamma1_5b, b_beta1_5b, W_gamma2_5b, b_beta2_5b,
                   W_norm_gamma_5b, b_norm_beta_5b,
                   W_fusion_1x1_5b, b_fusion_1x1_5b,
                   W_fusion_3x3_5b, b_fusion_3x3_5b,
                   W_gamma_f_1x1_5b, b_beta_f_1x1_5b,
                   W_gamma_f_3x3_5b, b_beta_f_3x3_5b,
                   # Head
                   W_head, b_head, W_gamma_head, b_beta_head,
                   # Classifier
                   W_cls, b_cls):
    """
    Full MobileViT-XXS forward pass — pure NumPy.

    Args:
        Inputs    : (B, 3, H, W) — input images, float32
        bn_prams  : list of 64 numpy arrays (alternating running_mean, running_var)
                    See bn_prams layout in extract_all_weights() docstring.
        **kwargs  : all weight and bias tensors (unpacked via **weights from extract_all_weights)
    Returns:
        logits : (B, 1000) — raw ImageNet class scores
    """
    # ── Stem: 3×3 Conv → BN → Swish ──────────────────────────────────────────
    # (B, 3, 224, 224) → (B, 16, 112, 112)
    x = conv_forward(Inputs, W_stem, b_stem, 2, 1)
    x = batch_norm_forward(x, W_stem_gamma, b_beta_stem, bn_prams[0], bn_prams[1])
    x = swish(x)

    # ── Stage 1: single MV2, no downsampling ─────────────────────────────────
    # (B, 16, 112, 112) → (B, 16, 112, 112) [residual]
    x = MobileNet_v2(x, W_expan_1, W_depth_1, W_point_1,
                     b_expan_1, b_depth_1, b_point_1, 1,
                     W_gamma_expan_1, W_gamma_depth_1, W_gamma_point_1,
                     b_gamma_expan_1, b_gamma_depth_1, b_gamma_point_1,
                     *bn_prams[2:8])

    # ── Stage 2: 1 downsampling MV2 + 2 residual MV2 ─────────────────────────
    # 2a: (B, 16, 112, 112) → (B, 24, 56, 56) [stride=2]
    x = MobileNet_v2(x, W_expan_2a, W_depth_2a, W_point_2a,
                     b_expan_2a, b_depth_2a, b_point_2a, 2,
                     W_gamma_expan_2a, W_gamma_depth_2a, W_gamma_point_2a,
                     b_gamma_expan_2a, b_gamma_depth_2a, b_gamma_point_2a,
                     *bn_prams[8:14])
    # 2b: (B, 24, 56, 56) → (B, 24, 56, 56) [residual]
    x = MobileNet_v2(x, W_expan_2b, W_depth_2b, W_point_2b,
                     b_expan_2b, b_depth_2b, b_point_2b, 1,
                     W_gamma_expan_2b, W_gamma_depth_2b, W_gamma_point_2b,
                     b_gamma_expan_2b, b_gamma_depth_2b, b_gamma_point_2b,
                     *bn_prams[14:20])
    # 2c: (B, 24, 56, 56) → (B, 24, 56, 56) [residual]
    x = MobileNet_v2(x, W_expan_2c, W_depth_2c, W_point_2c,
                     b_expan_2c, b_depth_2c, b_point_2c, 1,
                     W_gamma_expan_2c, W_gamma_depth_2c, W_gamma_point_2c,
                     b_gamma_expan_2c, b_gamma_depth_2c, b_gamma_point_2c,
                     *bn_prams[20:26])

    # ── Stage 3: MV2 downsample + MobileViT Block ─────────────────────────────
    # 3a: (B, 24, 56, 56) → (B, 48, 28, 28) [stride=2]
    x = MobileNet_v2(x, W_expan_3a, W_depth_3a, W_point_3a,
                     b_expan_3a, b_depth_3a, b_point_3a, 2,
                     W_gamma_expan_3a, W_gamma_depth_3a, W_gamma_point_3a,
                     b_gamma_expan_3a, b_gamma_depth_3a, b_gamma_point_3a,
                     *bn_prams[26:32])
    # 3b: (B, 48, 28, 28) → (B, 48, 28, 28)  d=64, L=2 transformers, H=2 heads
    x = MobileViTBlock_(x,
                        W_loacal_3x3_3b, b_local_3x3_3b,
                        W_loacal_1x1_3b, b_local_1x1_3b,
                        W_gamma_local_3x3_3b, W_beta_local_3x3_3b,
                        W_QKV_atten_3b, W_o_atten_3b, Bias_QKV_atten_3b, Bias_o_atten_3b,
                        W1_fc1_3b, b1_fc1_3b, W2_fc2_3b, b2_fc2_3b,
                        W_gamma1_3b, b_beta1_3b, W_gamma2_3b, b_beta2_3b,
                        W_norm_gamma_3b, b_norm_beta_3b,
                        W_fusion_1x1_3b, b_fusion_1x1_3b,
                        W_fusion_3x3_3b, b_fusion_3x3_3b,
                        W_gamma_f_1x1_3b, b_beta_f_1x1_3b,
                        W_gamma_f_3x3_3b, b_beta_f_3x3_3b,
                        2, 2, 2,           # L=2 layers, patch_size=2, num_heads=2
                        *bn_prams[32:38])

    # ── Stage 4: MV2 downsample + MobileViT Block ─────────────────────────────
    # 4a: (B, 48, 28, 28) → (B, 64, 14, 14) [stride=2]
    x = MobileNet_v2(x, W_expan_4a, W_depth_4a, W_point_4a,
                     b_expan_4a, b_depth_4a, b_point_4a, 2,
                     W_gamma_expan_4a, W_gamma_depth_4a, W_gamma_point_4a,
                     b_gamma_expan_4a, b_gamma_depth_4a, b_gamma_point_4a,
                     *bn_prams[38:44])
    # 4b: (B, 64, 14, 14) → (B, 64, 14, 14)  d=80, L=4 transformers, H=4 heads
    x = MobileViTBlock_(x,
                        W_loacal_3x3_4b, b_local_3x3_4b,
                        W_loacal_1x1_4b, b_local_1x1_4b,
                        W_gamma_local_3x3_4b, W_beta_local_3x3_4b,
                        W_QKV_atten_4b, W_o_atten_4b, Bias_QKV_atten_4b, Bias_o_atten_4b,
                        W1_fc1_4b, b1_fc1_4b, W2_fc2_4b, b2_fc2_4b,
                        W_gamma1_4b, b_beta1_4b, W_gamma2_4b, b_beta2_4b,
                        W_norm_gamma_4b, b_norm_beta_4b,
                        W_fusion_1x1_4b, b_fusion_1x1_4b,
                        W_fusion_3x3_4b, b_fusion_3x3_4b,
                        W_gamma_f_1x1_4b, b_beta_f_1x1_4b,
                        W_gamma_f_3x3_4b, b_beta_f_3x3_4b,
                        4, 2, 4,           # L=4 layers, patch_size=2, num_heads=4
                        *bn_prams[44:50])

    # ── Stage 5: MV2 downsample + MobileViT Block ─────────────────────────────
    # 5a: (B, 64, 14, 14) → (B, 80, 7, 7) [stride=2]
    x = MobileNet_v2(x, W_expan_5a, W_depth_5a, W_point_5a,
                     b_expan_5a, b_depth_5a, b_point_5a, 2,
                     W_gamma_expan_5a, W_gamma_depth_5a, W_gamma_point_5a,
                     b_gamma_expan_5a, b_gamma_depth_5a, b_gamma_point_5a,
                     *bn_prams[50:56])
    # 5b: (B, 80, 7, 7) → (B, 80, 7, 7)   d=96, L=3 transformers, H=4 heads
    x = MobileViTBlock_(x,
                        W_loacal_3x3_5b, b_local_3x3_5b,
                        W_loacal_1x1_5b, b_local_1x1_5b,
                        W_gamma_local_3x3_5b, W_beta_local_3x3_5b,
                        W_QKV_atten_5b, W_o_atten_5b, Bias_QKV_atten_5b, Bias_o_atten_5b,
                        W1_fc1_5b, b1_fc1_5b, W2_fc2_5b, b2_fc2_5b,
                        W_gamma1_5b, b_beta1_5b, W_gamma2_5b, b_beta2_5b,
                        W_norm_gamma_5b, b_norm_beta_5b,
                        W_fusion_1x1_5b, b_fusion_1x1_5b,
                        W_fusion_3x3_5b, b_fusion_3x3_5b,
                        W_gamma_f_1x1_5b, b_beta_f_1x1_5b,
                        W_gamma_f_3x3_5b, b_beta_f_3x3_5b,
                        3, 2, 4,           # L=3 layers, patch_size=2, num_heads=4
                        *bn_prams[56:62])

    # ── Head ─────────────────────────────────────────────────────────────────
    # (B, 80, 7, 7) → (B, 320, 7, 7)
    x = conv_forward(x, W_head, b_head, 1, 0)
    x = batch_norm_forward(x, W_gamma_head, b_beta_head, bn_prams[62], bn_prams[63])
    x = swish(x)

    # Global average pool: (B, 320, 7, 7) → (B, 320)
    x = global_avg_pool(x)

    # Classifier: W_cls is (1000, 320) — transpose for matmul
    W_cls = W_cls.T   # → (320, 1000)
    return linear_classifier(x, W_cls, b_cls)   # → (B, 1000)


print("✅ MobileViT_XXS_ (NumPy) defined")
print("   178 weight tensors + 64 BN running-stat arrays accepted")


✅ MobileViT_XXS_ (NumPy) defined
   178 weight tensors + 64 BN running-stat arrays accepted


---

## 🔄 Section 4 — Weight Extraction: PyTorch → NumPy

The extraction functions convert every PyTorch tensor in our weight-mapped model  
to NumPy arrays in the exact format expected by `MobileViT_XXS_()`.

### Tensor Layout Notes

| Tensor Type | PyTorch Shape | NumPy Shape | Notes |
|-------------|---------------|-------------|-------|
| Standard conv weight | `(C_out, C_in, f, f)` | same | ✓ |
| Depthwise conv weight | `(C, 1, f, f)` | `(C, f, f)` | squeezed |
| Conv bias (bias=False) | — | `(C_out, 1, 1, 1)` | zeros |
| BN γ/β | `(C,)` | `(C,)` | direct |
| BN running stats | `(C,)` | stored in `bn_prams[]` | separate list |
| QKV weight | `in_proj_weight (3d, d)` | same | ✓ (already concatenated) |
| MLP weight | `(d_ff, d_model)` | same | NumPy does `.T` internally |
| Classifier | `(1000, 320)` | same | NumPy does `.T` internally |

### BN Running Stats Layout (64 entries)

```
[ 0, 1]  stem          — mean, var
[ 2..7]  mv2_1         — expand (mean,var), depthwise (mean,var), project (mean,var)
[ 8..13] mv2_2a
[14..19] mv2_2b
[20..25] mv2_2c
[26..31] mv2_3a
[32..37] mvit_3b       — local-3×3 (mean,var), fusion-1×1 (mean,var), fusion-3×3 (mean,var)
[38..43] mv2_4a
[44..49] mvit_4b
[50..55] mv2_5a
[56..61] mvit_5b
[62,63]  head conv
```


In [ ]:
## ============================================================
## Weight Extraction: PyTorch Model → NumPy Arrays
## ============================================================
##
## extract_all_weights() collects all 178 weight tensors and 64 BN
## running-stat arrays from the PyTorch model. The result is passed
## directly into MobileViT_XXS_() for NumPy inference.
## ============================================================

import numpy as np


# ── Conversion helpers ────────────────────────────────────────────────────────

def t(tensor):
    """PyTorch tensor → NumPy float32."""
    return tensor.detach().cpu().numpy().astype(np.float32)

def zero_conv_bias(conv):
    """Create zero bias shaped (C_out, 1, 1, 1) for a bias=False conv layer."""
    return np.zeros((conv.weight.shape[0], 1, 1, 1), dtype=np.float32)

def dw_weight(conv):
    """Squeeze depthwise weight from (C, 1, f, f) → (C, f, f)."""
    return t(conv.weight).squeeze(1)

def zero_dw_bias(conv):
    """Create zero bias shaped (C,) for a bias=False depthwise conv."""
    return np.zeros(conv.weight.shape[0], dtype=np.float32)


# ── MV2 Block Extractor ───────────────────────────────────────────────────────
def extract_mv2(block, suffix):
    """
    Extract all weights and BN running stats from one MV2 block.

    Returns
    -------
    weights  : dict of 12 arrays (expand/depthwise/project: W, b, γ, β)
    bn_stats : list of 6 arrays  [mean1, var1, mean2, var2, mean3, var3]
    """
    conv_e, bn_e = block.pw_expand[0],  block.pw_expand[1]
    conv_d, bn_d = block.dw_conv[0],    block.dw_conv[1]
    conv_p, bn_p = block.pw_project[0], block.pw_project[1]

    weights = {
        f'W_expan_{suffix}':        t(conv_e.weight),
        f'b_expan_{suffix}':        zero_conv_bias(conv_e),
        f'W_gamma_expan_{suffix}':  t(bn_e.weight),
        f'b_gamma_expan_{suffix}':  t(bn_e.bias),
        f'W_depth_{suffix}':        dw_weight(conv_d),
        f'b_depth_{suffix}':        zero_dw_bias(conv_d),
        f'W_gamma_depth_{suffix}':  t(bn_d.weight),
        f'b_gamma_depth_{suffix}':  t(bn_d.bias),
        f'W_point_{suffix}':        t(conv_p.weight),
        f'b_point_{suffix}':        zero_conv_bias(conv_p),
        f'W_gamma_point_{suffix}':  t(bn_p.weight),
        f'b_gamma_point_{suffix}':  t(bn_p.bias),
    }
    bn_stats = [
        t(bn_e.running_mean), t(bn_e.running_var),
        t(bn_d.running_mean), t(bn_d.running_var),
        t(bn_p.running_mean), t(bn_p.running_var),
    ]
    return weights, bn_stats


# ── MobileViT Block Extractor ─────────────────────────────────────────────────
def extract_mvit(block, suffix):
    """
    Extract all weights and BN running stats from one MobileViTBlock.

    Transformer layers: QKV weight is already concatenated (in_proj_weight).
    MLP weights are extracted as-is (NumPy applies .T internally).

    Returns
    -------
    weights  : dict (local rep + all transformer layers + fusion)
    bn_stats : list of 6 arrays [local-3×3 mean/var, fusion-1×1 mean/var, fusion-3×3 mean/var]
    """
    lr  = block.local_rep
    fus = block.fusion
    conv3, bn3 = lr.conv1, lr.bn1
    conv1       = lr.conv2  # no BN on 1×1

    weights = {
        f'W_loacal_3x3_{suffix}':      t(conv3.weight),
        f'b_local_3x3_{suffix}':       zero_conv_bias(conv3),
        f'W_gamma_local_3x3_{suffix}': t(bn3.weight),
        f'W_beta_local_3x3_{suffix}':  t(bn3.bias),
        f'W_loacal_1x1_{suffix}':      t(conv1.weight),
        f'b_local_1x1_{suffix}':       zero_conv_bias(conv1),
    }
    bn_stats = [t(bn3.running_mean), t(bn3.running_var)]  # local-3×3 BN

    # Collect per-layer transformer weights into lists
    # (each list has L entries, one per transformer layer)
    lists = {k: [] for k in [
        'W_QKV_atten', 'Bias_QKV_atten', 'W_o_atten', 'Bias_o_atten',
        'W_gamma1', 'b_beta1', 'W_gamma2', 'b_beta2',
        'W1_fc1', 'b1_fc1', 'W2_fc2', 'b2_fc2',
    ]}
    for layer in block.transformers:
        lists['W_QKV_atten'].append(t(layer.attn.in_proj_weight))  # (3d, d)
        lists['Bias_QKV_atten'].append(t(layer.attn.in_proj_bias)) # (3d,)
        lists['W_o_atten'].append(t(layer.attn.out_proj.weight))   # (d, d)
        lists['Bias_o_atten'].append(t(layer.attn.out_proj.bias))  # (d,)
        lists['W_gamma1'].append(t(layer.norm1.weight))
        lists['b_beta1'].append(t(layer.norm1.bias))
        lists['W_gamma2'].append(t(layer.norm2.weight))
        lists['b_beta2'].append(t(layer.norm2.bias))
        lists['W1_fc1'].append(t(layer.fc1.weight))   # (d_ff, d)
        lists['b1_fc1'].append(t(layer.fc1.bias))
        lists['W2_fc2'].append(t(layer.fc2.weight))   # (d, d_ff)
        lists['b2_fc2'].append(t(layer.fc2.bias))
    for key, val in lists.items():
        weights[f'{key}_{suffix}'] = val

    # Final LayerNorm (applied after the full transformer stack)
    weights[f'W_norm_gamma_{suffix}'] = t(block.norm.weight)
    weights[f'b_norm_beta_{suffix}']  = t(block.norm.bias)

    # Fusion conv weights + BN
    fc1, fbn1 = fus.conv1, fus.bn1
    fc2, fbn2 = fus.conv2, fus.bn2
    weights[f'W_fusion_1x1_{suffix}']   = t(fc1.weight)
    weights[f'b_fusion_1x1_{suffix}']   = zero_conv_bias(fc1)
    weights[f'W_gamma_f_1x1_{suffix}']  = t(fbn1.weight)
    weights[f'b_beta_f_1x1_{suffix}']   = t(fbn1.bias)
    weights[f'W_fusion_3x3_{suffix}']   = t(fc2.weight)
    weights[f'b_fusion_3x3_{suffix}']   = zero_conv_bias(fc2)
    weights[f'W_gamma_f_3x3_{suffix}']  = t(fbn2.weight)
    weights[f'b_beta_f_3x3_{suffix}']   = t(fbn2.bias)
    bn_stats += [
        t(fbn1.running_mean), t(fbn1.running_var),  # fusion-1×1 BN
        t(fbn2.running_mean), t(fbn2.running_var),  # fusion-3×3 BN
    ]
    return weights, bn_stats  # 6 BN entries per MobileViTBlock


# ── Main Extraction Function ──────────────────────────────────────────────────
def extract_all_weights(model):
    """
    Extract all parameters from MobileViT_XXS (PyTorch) → NumPy arrays.

    Parameters
    ----------
    model : MobileViT_XXS, must be in eval() mode

    Returns
    -------
    weights  : dict  — 178 NumPy arrays, unpack with **weights into MobileViT_XXS_()
    bn_prams : list  — 64 NumPy arrays (alternating running_mean, running_var)
    """
    model.eval()
    weights  = {}
    bn_prams = []

    # Stem
    conv_s, bn_s = model.stem[0], model.stem[1]
    weights.update({'W_stem': t(conv_s.weight), 'b_stem': zero_conv_bias(conv_s),
                    'W_stem_gamma': t(bn_s.weight), 'b_beta_stem': t(bn_s.bias)})
    bn_prams += [t(bn_s.running_mean), t(bn_s.running_var)]   # [0, 1]

    # Stage 1
    w, bn = extract_mv2(model.mv2_1, '1')
    weights.update(w); bn_prams += bn                          # [2..7]

    # Stage 2
    for blk, suf in [(model.mv2_2a, '2a'), (model.mv2_2b, '2b'), (model.mv2_2c, '2c')]:
        w, bn = extract_mv2(blk, suf)
        weights.update(w); bn_prams += bn                      # [8..25]

    # Stage 3
    w, bn = extract_mv2(model.mv2_3a, '3a')
    weights.update(w); bn_prams += bn                          # [26..31]
    w, bn = extract_mvit(model.mvit_3b, '3b')
    weights.update(w); bn_prams += bn                          # [32..37]

    # Stage 4
    w, bn = extract_mv2(model.mv2_4a, '4a')
    weights.update(w); bn_prams += bn                          # [38..43]
    w, bn = extract_mvit(model.mvit_4b, '4b')
    weights.update(w); bn_prams += bn                          # [44..49]

    # Stage 5
    w, bn = extract_mv2(model.mv2_5a, '5a')
    weights.update(w); bn_prams += bn                          # [50..55]
    w, bn = extract_mvit(model.mvit_5b, '5b')
    weights.update(w); bn_prams += bn                          # [56..61]

    # Head conv
    conv_h, bn_h = model.head_conv[0], model.head_conv[1]
    weights.update({'W_head': t(conv_h.weight), 'b_head': zero_conv_bias(conv_h),
                    'W_gamma_head': t(bn_h.weight), 'b_beta_head': t(bn_h.bias)})
    bn_prams += [t(bn_h.running_mean), t(bn_h.running_var)]   # [62, 63]

    # Classifier (W_cls is (1000, 320); NumPy will transpose internally)
    weights['W_cls'] = t(model.classifier.weight)
    weights['b_cls'] = t(model.classifier.bias)

    assert len(bn_prams) == 64, f"Expected 64 bn_prams, got {len(bn_prams)}"
    return weights, bn_prams


print("✅ Weight extraction functions defined: extract_mv2, extract_mvit, extract_all_weights")


✅ Weight extraction functions defined: extract_mv2, extract_mvit, extract_all_weights


---

## ✅ Section 5 — Numerical Validation

This is the final proof-of-concept: run both the PyTorch model and the NumPy engine  
on **identical input**, then compare every output logit.

### What We're Checking

1. **Shapes match** — both produce `(1, 1000)` logits
2. **Values agree** — max absolute difference < 5×10⁻⁴
3. **Rank agreement** — top-5 predicted classes are identical

Passing this test confirms that:
- The weight mapping is correct (all 178 tensors landed in the right places)
- The BN running stats are correctly routed (all 64 entries)
- The unfold/fold cycle is numerically invertible
- The attention computation matches PyTorch's `nn.MultiheadAttention`
- The float32 accumulation errors are within acceptable hardware tolerance


In [ ]:
import torch
import numpy as np

# ── Extract weights from the pretrained PyTorch model ────────────────────────
# This call traverses every module in my_model and converts its parameters
# to NumPy arrays in the exact format MobileViT_XXS_() expects.
torch.manual_seed(0)
weights, bn_prams = extract_all_weights(my_model)

print("=" * 60)
print(f"  Total weight tensors : {len(weights)}")
print(f"  bn_prams entries     : {len(bn_prams)}")
print()

# Print a summary of each weight tensor's shape
print("Weight tensor shapes:")
for k, v in weights.items():
    if isinstance(v, list):
        print(f"  {k}: list ×{len(v)}, each {v[0].shape}")
    else:
        print(f"  {k}: {v.shape}")

print()
print("BN running-stat shapes:")
for i, b in enumerate(bn_prams):
    print(f"  [{i:2d}] {b.shape}")


  Total weight tensors : 178
  bn_prams entries     : 64

Weight tensor shapes:
  W_stem: (16, 3, 3, 3)
  b_stem: (16, 1, 1, 1)
  W_stem_gamma: (16,)
  b_beta_stem: (16,)
  W_expan_1: (32, 16, 1, 1)
  b_expan_1: (32, 1, 1, 1)
  W_gamma_expan_1: (32,)
  b_gamma_expan_1: (32,)
  W_depth_1: (32, 3, 3)
  b_depth_1: (32,)
  W_gamma_depth_1: (32,)
  b_gamma_depth_1: (32,)
  W_point_1: (16, 32, 1, 1)
  b_point_1: (16, 1, 1, 1)
  W_gamma_point_1: (16,)
  b_gamma_point_1: (16,)
  W_expan_2a: (32, 16, 1, 1)
  b_expan_2a: (32, 1, 1, 1)
  W_gamma_expan_2a: (32,)
  b_gamma_expan_2a: (32,)
  W_depth_2a: (32, 3, 3)
  b_depth_2a: (32,)
  W_gamma_depth_2a: (32,)
  b_gamma_depth_2a: (32,)
  W_point_2a: (24, 32, 1, 1)
  b_point_2a: (24, 1, 1, 1)
  W_gamma_point_2a: (24,)
  b_gamma_point_2a: (24,)
  W_expan_2b: (48, 24, 1, 1)
  b_expan_2b: (48, 1, 1, 1)
  W_gamma_expan_2b: (48,)
  b_gamma_expan_2b: (48,)
  W_depth_2b: (48, 3, 3)
  b_depth_2b: (48,)
  W_gamma_depth_2b: (48,)
  b_gamma_depth_2b: (48,)
  W_p

In [ ]:
# ── Numerical comparison: PyTorch vs NumPy ───────────────────────────────────
print("=" * 60)
print("Numerical validation — same random input through both engines")
print("=" * 60)

np.random.seed(42)
x_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
x_pt = torch.from_numpy(x_np)

# PyTorch forward pass
with torch.no_grad():
    y_pt = my_model(x_pt).numpy()

# NumPy forward pass
y_np = MobileViT_XXS_(x_np, bn_prams, **weights)

max_diff  = np.abs(y_pt - y_np).max()
mean_diff = np.abs(y_pt - y_np).mean()

print(f"\n  PyTorch output shape : {y_pt.shape}")
print(f"  NumPy   output shape : {y_np.shape}")
print(f"\n  PyTorch first 5 logits : {y_pt[0,:5]}")
print(f"  NumPy   first 5 logits : {y_np[0,:5]}")
print(f"\n  Max  absolute diff : {max_diff:.6e}")
print(f"  Mean absolute diff : {mean_diff:.6e}")

threshold = 5e-4
if max_diff < threshold:
    print(f"\n  ✅ PASS — outputs match within {threshold:.0e}")
else:
    print(f"\n  ❌ FAIL — max diff {max_diff:.2e} exceeds threshold {threshold:.0e}")

# Top-5 class agreement
top5_pt = np.argsort(y_pt[0])[-5:][::-1]
top5_np = np.argsort(y_np[0])[-5:][::-1]
print(f"\n  Top-5 PyTorch classes : {top5_pt}")
print(f"  Top-5 NumPy   classes : {top5_np}")
print(f"  Top-5 match           : {'✅ YES' if list(top5_pt) == list(top5_np) else '❌ NO'}")


Numerical validation — same random input through both engines

  PyTorch output shape : (1, 1000)
  NumPy   output shape : (1, 1000)

  PyTorch first 5 logits : [-458.5755  -291.0059  -278.06158 -414.248   -359.8885 ]
  NumPy   first 5 logits : [-458.57561839 -291.00589838 -278.06167292 -414.24807382 -359.88869868]

  Max  absolute diff : 3.070984e-04
  Mean absolute diff : 9.771453e-05

  ✅ PASS — outputs match within 5e-04

  Top-5 PyTorch classes : [852 626 711 867 605]
  Top-5 NumPy   classes : [852 626 711 867 605]
  Top-5 match           : ✅ YES
